In [59]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.pyplot as plt
import seaborn as sns

# Для отображения графиков в ноутбуке
%matplotlib inline

Датасет возьмем снова KION: https://github.com/irsafilo/KION_DATASET/

In [60]:
items = pd.read_csv('../../items.csv')
items.head(1)

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."


In [62]:
df = pd.read_csv('../../interactions.csv')
df.head()

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250.0,72.0
1,699317,1659,2021-05-29,8317.0,100.0
2,656683,7107,2021-05-09,10.0,0.0
3,864613,7638,2021-07-05,14483.0,100.0
4,964868,9506,2021-04-30,6725.0,100.0


In [63]:
most_pop_recs = df.item_id.value_counts()[:10].index.values.tolist()
most_pop_recs

[10440, 15297, 9728, 13865, 4151, 3734, 2657, 4880, 142, 6809]

In [66]:
random_recs = df.item_id.sample(n=10).values.tolist()
random_recs

[16289, 14942, 14051, 9359, 8922, 12360, 2942, 15297, 3411, 9927]

In [67]:
novel_recs = df.item_id.value_counts()[-10:].index.values.tolist()
novel_recs

[11315, 9467, 11704, 14528, 1254, 1506, 6512, 5332, 4109, 11091]

In [68]:
test_users = df.user_id.sample(n=1000).reset_index()
test_users['index'] = 1

most_pop_recs = pd.DataFrame(most_pop_recs, columns=['item_id'])
random_recs = pd.DataFrame(random_recs, columns=['item_id'])
novel_recs = pd.DataFrame(novel_recs, columns=['item_id'])

most_pop_recs['index'] = 1
random_recs['index'] = 1
novel_recs['index'] = 1

most_pop_data = test_users.merge(most_pop_recs).drop(['index'], axis=1)
random_data = test_users.merge(random_recs).drop(['index'], axis=1)
novel_recs_data = test_users.merge(novel_recs).drop(['index'], axis=1)

In [69]:
most_pop_data.head()

,user_id,item_id
0,333829,10440
1,333829,15297
2,333829,9728
3,333829,13865
4,333829,4151


In [70]:
random_data['item_id'] = df.item_id.sample(n=random_data.shape[0], replace=True).values
random_data.head()

,user_id,item_id
0,333829,6384
1,333829,15297
2,333829,11605
3,333829,12118
4,333829,10440


In [71]:
novel_recs_data.head()

,user_id,item_id
0,333829,11315
1,333829,9467
2,333829,11704
3,333829,14528
4,333829,1254


## Новизна

Novelty измеряет, насколько новые/непопулярные товары рекомендуются пользователям.

In [72]:
# Рассчитаем популярность каждого товара (сколько пользователей его смотрели)
item_popularity = df.groupby('item_id')['user_id'].nunique().reset_index()
item_popularity.columns = ['item_id', 'num_users']

# Функция для расчета novelty рекомендаций
def calculate_novelty(recommendations, item_popularity):
    """
    recommendations - DataFrame с колонками: user_id, item_id
    item_popularity - DataFrame с колонками: item_id, num_users
    """
    rec_with_pop = pd.merge(recommendations, item_popularity, on='item_id', how='left')
    total_users = recommendations['user_id'].nunique()
    novelty = -np.log(rec_with_pop['num_users'] / total_users).mean()
    return novelty


for x in [most_pop_data, random_data, novel_recs_data]:
    novelty_score = calculate_novelty(x, item_popularity)
    print(f"Novelty score: {novelty_score:.4f}")

Novelty score: -3.2509
Novelty score: -0.3014
Novelty score: 6.9058


## Покрытие (coverage)

In [73]:
def calculate_catalog_coverage(recommendations, catalog_size):
    """
    recommendations - DataFrame с колонками: user_id, item_id
    catalog_size - общее количество товаров в каталоге
    """
    unique_recommended_items = recommendations['item_id'].nunique()
    coverage = unique_recommended_items / catalog_size
    return coverage

# Пример использования
catalog_size = df['item_id'].nunique()  # в реальности может быть больше

for x in [most_pop_data, random_data, novel_recs_data]:
    coverage_score = calculate_catalog_coverage(x, catalog_size)
    print(f"Catalog coverage: {coverage_score:.2%}")


Catalog coverage: 0.08%
Catalog coverage: 20.48%
Catalog coverage: 0.08%


## Разнообразие

In [74]:
from tqdm import tqdm

item_sim_flag = {}
for row1 in tqdm(items.itertuples()):
    for row2 in items[row1.Index:].itertuples():
        if row1.item_id != row2.item_id:
            if row1.genres == row2.genres:
                item_sim_flag[(row1.item_id, row2.item_id)] = 1


15963it [01:32, 171.67it/s] 


In [46]:
item_sim_flag

{(10711, 9182): 1,
 (10711, 15851): 1,
 (10711, 9224): 1,
 (10711, 14509): 1,
 (2508, 1245): 1,
 (2508, 7131): 1,
 (2508, 16507): 1,
 (2508, 12576): 1,
 (2508, 12665): 1,
 (2508, 15288): 1,
 (2508, 16481): 1,
 (2508, 3311): 1,
 (2508, 2503): 1,
 (2508, 10478): 1,
 (2508, 13637): 1,
 (2508, 8004): 1,
 (10716, 5097): 1,
 (10716, 4900): 1,
 (10716, 7945): 1,
 (10716, 888): 1,
 (10716, 3797): 1,
 (10716, 8395): 1,
 (7868, 13729): 1,
 (7868, 6521): 1,
 (7868, 10460): 1,
 (7868, 12445): 1,
 (7868, 9177): 1,
 (7868, 9971): 1,
 (7868, 5548): 1,
 (7868, 16206): 1,
 (7868, 5707): 1,
 (7868, 168): 1,
 (7868, 1806): 1,
 (7868, 555): 1,
 (7868, 6480): 1,
 (7868, 5368): 1,
 (7868, 6490): 1,
 (7868, 4170): 1,
 (7868, 15869): 1,
 (7868, 6616): 1,
 (7868, 3997): 1,
 (7868, 5478): 1,
 (7868, 5001): 1,
 (7868, 14092): 1,
 (7868, 15990): 1,
 (7868, 11299): 1,
 (7868, 15539): 1,
 (7868, 5480): 1,
 (7868, 14650): 1,
 (7868, 16404): 1,
 (7868, 4325): 1,
 (7868, 2169): 1,
 (7868, 7423): 1,
 (7868, 2067): 1,
 

In [75]:
def calculate_intra_list_diversity(user_recommendations, item_similarity):
    """
    Рассчитывает разнообразие рекомендаций для одного пользователя
    """
    n = len(user_recommendations)
    if n < 2:
        return 1.0  # максимальное разнообразие для одного элемента
    
    total = 0
    count = 0
    for i in range(n):
        for j in range(i+1, n):
            item_i = user_recommendations[i]
            item_j = user_recommendations[j]
            total += 1 - item_sim_flag.get((item_i, item_j), 0)
            count += 1
    
    return total / count if count > 0 else 1.0

# Пример расчета для одного пользователя


# Пример использования
catalog_size = df['item_id'].nunique()  # в реальности может быть больше

for x in [most_pop_data, random_data, novel_recs_data]:
    user_id = x.user_id.sample(n=1).values[0]
    user_recommendations = x[x['user_id'] == user_id]['item_id'].tolist()
    diversity_score = calculate_intra_list_diversity(user_recommendations, item_sim_flag)
    print(f"Intra-list diversity for user 1: {diversity_score:.4f}")


Intra-list diversity for user 1: 0.9556
Intra-list diversity for user 1: 1.0000
Intra-list diversity for user 1: 1.0000


In [76]:
def calculate_inter_list_diversity(recommendations, max_count = 100):
    """
    Рассчитывает разнообразие рекомендаций между пользователями
    """
    user_recs = recommendations.groupby('user_id')['item_id'].apply(list).reset_index()
    rec_lists = user_recs['item_id'].tolist()
    
    if len(rec_lists) < 2:
        return 1.0
    
    total = 0
    count = 0
    for i in range(len(rec_lists)):
        for j in range(i+1, len(rec_lists)):
            set_i = set(rec_lists[i])
            set_j = set(rec_lists[j])
            intersection = len(set_i & set_j)
            union = len(set_i | set_j)
            jaccard_sim = intersection / union if union > 0 else 0
            total += 1 - jaccard_sim
            count += 1
    
    return total / count if count > 0 else 1.0

# Пример расчета
for x in [most_pop_data, random_data, novel_recs_data]:
    inter_diversity_score = calculate_inter_list_diversity(x)
    print(f"Inter-list diversity: {inter_diversity_score:.4f}")

Inter-list diversity: 0.0000
Inter-list diversity: 0.9765
Inter-list diversity: 0.0000
